# Track 1 · Data handoff for detection, segmentation and integration (one-time, before 3 Nov 2026)

AI251P Capstone. Owner: Abdulmohsen Alharthi. Packages the Track 1 data **once** for Abdulelah (detection), Ziyad (segmentation) and Rayan (integration).

- Read-only on every existing file. No training, no model, no predictions, no W&B, no secrets. Labels only, no test evaluation, no TestLog entry.
- Conversion to a detector format is not in scope (detection track).
- One Drive root: `DRIVE_PROJECT`. Every path written into an output file is relative to it.
- A missing input stops the notebook and is printed; nothing is guessed or rebuilt.
- Output: `01_Data/Handoff/<YYYY-MM-DD>_Track1Handoff_vN/` (never overwritten; a re-run makes the next vN).
- Every check is a hard assert (stops at the first failure) and is written to `validation_report.json` as expected / actual / pass.

Runtime: CPU is enough.

## 0. Settings, mount Drive

In [ ]:
DRIVE_PROJECT = '/content/drive/MyDrive/AI251P_Capstone_CrossDomainAerial'   # the only Drive path
PHASE1_COMMIT = '7a229818876f6595a9acf0854af756b9d3b4acea'
CLASSES = ['swimmer', 'boat', 'jetski', 'life_saving_appliances', 'buoy', 'car', 'truck', 'bus']   # never re-sorted
DOMAIN = {c: ('sea' if i < 5 else 'urban') for i, c in enumerate(CLASSES)}
SPLITS = ['train', 'val', 'test']

IN = {   # all relative to DRIVE_PROJECT, read-only
    'splits':       '01_Data/Processed/Splits/2026-10-19_Splits_v1.json',
    'class_crops':  '01_Data/Processed/Splits/2026-10-19_ClassCrops_v1.csv.gz',
    'split_table':  '01_Data/Processed/Splits/2026-10-19_SplitClassTable_v1.csv',
    'sds_clean':    '01_Data/Processed/SeaDronesSee/2026-10-18_SDSClean_v1.json',
    'uav_frames':   '01_Data/Processed/UAVDT/2026-10-18_UAVDTFrames_v1.csv',
    'crop_index':   '01_Data/Processed/Crops/2026-10-20_CropIndex_v1.csv.gz',
    'npy_train':    '01_Data/Processed/Crops/2026-10-20_CropsTrain_v1.npy',
    'npy_val':      '01_Data/Processed/Crops/2026-10-20_CropsVal_v1.npy',
    'npy_test':     '01_Data/Processed/Crops/2026-10-20_CropsTest_v1.npy',
    'ckpt':         '02_Classification/Checkpoints/2026-10-23_cls-full-64-s42-lr0.002_v3.pt',
    'ah_manifest':  '01_Data/Team_AlHada/2026-10-19_AlHadaFramesManifest_v1.csv',
    'ah_frames':    '01_Data/Team_AlHada/Frames',
    'ah_zip_c02a':  '01_Data/Team_AlHada/Annotations/Annotations_CLIP02_t0075-0148_Ziyad_V2.zip',
    'ah_zip_c02b':  '01_Data/Team_AlHada/Annotations/Annotations_CLIP02_t0001-00074_Ziyad_V2.zip',
    'ah_zip_c01':   '01_Data/Team_AlHada/Annotations/task_2685723_annotations_CLIP01-T0001-T0164_coco 1.0RAYAN.zip',
}
PHASE1_BOXES = '02_Classification/2026-10-26_AlHadaPredictions_v2.csv.gz'   # Phase 1 per-box output: ONLY its box columns are read
if PHASE1_BOXES is not None: IN['ah_phase1_boxes'] = PHASE1_BOXES
AH_ZIPS = ['ah_zip_c02a', 'ah_zip_c02b', 'ah_zip_c01']           # same order as Phase 1 FILES (row order depends on it)
PHASE1_BOX_COLS = ['file', 'image', 'category', 'x', 'y', 'w', 'h', 'rotated', 'clip']   # prediction columns are never read
AH_SUPERSEDED = 'CLIP01-T0001-T0127.zip'          # partial export, never used

EXPECT = {
    'uav_frames': 40_733,   # Step 2 uavdt_after.frames_total: 40,735 on disk minus 2 exact duplicate frames
    'uav_frames_5fps': 6_807,   # Step 2 frames_5fps
    'uav_sequences': 50, 'sds_rgb_images': 1_338, 'rgb_crops': 5_549,   # dataset card
    'crops_total': 94_276, 'crops_per_split': {'train': 66_925, 'val': 13_800, 'test': 13_551},
    'ah_frames_per_clip': {'CLIP01': 157, 'CLIP02': 143}, 'ah_boxes': 9_484,
    'ah_boxes_per_clip_class': {'CLIP01': {'car': 7_611, 'truck': 402, 'bus': 47},
                                'CLIP02': {'car': 1_274, 'truck': 141, 'bus': 9}},
    'ah_boxes_per_class': {'car': 8_885, 'truck': 543, 'bus': 56},
    'ah_manifest_rows': 312, 'ah_excluded': 12, 'ah_image_size': (1920, 1080),
}
EXCLUDED = ([('CLIP01', t) for t in range(2, 9)] + [('CLIP02', t) for t in range(23, 28)])   # camera blocked by phone
EXCLUDED_REASON = 'camera blocked by phone'
assert len(EXCLUDED) == EXPECT['ah_excluded']
PREVIEW_N, PREVIEW_SEED = 10, 42

from google.colab import drive
drive.mount('/content/drive')

## 1. Inputs: all must exist (stop and print the missing ones)

In [ ]:
import os, re, io, json, math, time, shutil, hashlib, zipfile
from pathlib import Path
import numpy as np, pandas as pd
pd.set_option('display.width', 200)
ROOT = Path(DRIVE_PROJECT)
assert ROOT.is_dir(), f'DRIVE_PROJECT not found: {DRIVE_PROJECT}'
P = {k: ROOT / v for k, v in IN.items()}
missing = [f'{k}: {IN[k]}' for k, p in P.items() if not p.exists()]
if PHASE1_BOXES is None: missing.append('ah_phase1_boxes: PHASE1_BOXES not set in section 0')
if missing:
    print('MISSING INPUTS (nothing was written):'); print('\n'.join('  ' + m for m in missing))
    raise FileNotFoundError(f'{len(missing)} missing input(s), see the list above')
assert not any(AH_SUPERSEDED in v for v in IN.values()), 'the superseded CLIP01 export must not be an input'
try:
    from pycocotools.coco import COCO
except ImportError as e:
    raise ImportError('pycocotools is needed for the final COCO load check: pip install pycocotools') from e
print('all', len(P), 'inputs found')

REPORT = []
def check(name, expected, actual, ok=None):
    """hard assert; every check is logged as expected / actual / pass"""
    ok = (expected == actual) if ok is None else bool(ok)
    j = lambda v: v if isinstance(v, (str, int, float, bool, type(None), list, dict)) else str(v)
    REPORT.append({'check': name, 'expected': j(expected), 'actual': j(actual), 'pass': ok})
    print(f"{'PASS' if ok else 'FAIL'}  {name}: expected {expected}, actual {actual}")
    assert ok, f'check failed: {name} (expected {expected}, actual {actual})'

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 20), b''): h.update(b)
    return h.hexdigest()
def rel(p): return Path(p).relative_to(ROOT).as_posix()

## 1b. Al-Hada boxes: Phase 1 code, verbatim (reads only, writes nothing)

Copied verbatim from `notebooks/track1/alhada_eval_v2_cell_2026-10-26.py` (Phase 1 commit `7a22981`): `FILES` (line 9, order kept: the Phase 1 row order depends on it), `MAP` (line 11) and section `# 1) boxes` (lines 14–29: zip reading, CVAT category map, image id → file name, rotation → enclosing axis-aligned box, duplicate assert). Only `AN` is built from `DRIVE_PROJECT` (Phase 1 line 8 used `R`, the same folder). The next cell checks the result against the box columns of the Phase 1 output file `PHASE1_BOXES`; its prediction columns are never read.

In [ ]:
AN = f'{DRIVE_PROJECT}/01_Data/Team_AlHada/Annotations'     # Phase 1 line 8: AN = f'{R}/01_Data/Team_AlHada/Annotations', R = DRIVE_PROJECT
# ---- Phase 1 (alhada_eval_v2_cell_2026-10-26.py, commit 7a22981), verbatim from here ----
FILES = ['Annotations_CLIP02_t0075-0148_Ziyad_V2.zip', 'Annotations_CLIP02_t0001-00074_Ziyad_V2.zip', 'task_2685723_annotations_CLIP01-T0001-T0164_coco 1.0RAYAN.zip']
MAP = {'Car': 'car', 'Truck': 'truck', 'Bus': 'bus'}

# 1) boxes
rows = []
for f in FILES:
    z = zipfile.ZipFile(f'{AN}/{f}'); c = json.loads(z.read([n for n in z.namelist() if n.endswith('.json')][0]))
    cat = {x['id']: MAP[x['name']] for x in c['categories']}; img = {i['id']: i['file_name'] for i in c['images']}
    for a in c['annotations']:
        x, y, w, h = a['bbox']; t = math.radians((a.get('attributes') or {}).get('rotation', 0) or 0)
        W, H = abs(w*math.cos(t)) + abs(h*math.sin(t)), abs(w*math.sin(t)) + abs(h*math.cos(t)); cx, cy = x + w/2, y + h/2
        rows.append({'file': f, 'image': img[a['image_id']], 'category': cat[a['category_id']], 'x': cx-W/2, 'y': cy-H/2, 'w': W, 'h': H,
                     'rotated': t != 0})
B = pd.DataFrame(rows); B['clip'] = B.image.str.extract(r'(CLIP0\d)')[0]; B['box_side'] = B[['w','h']].max(axis=1)
assert not B.duplicated(['image','x','y','w','h']).any()
assert B.groupby('clip').image.nunique().to_dict() == {'CLIP01': 157, 'CLIP02': 143}, B.groupby('clip').image.nunique().to_dict()
assert len(B) == 9484, len(B)
print(B.groupby('clip').agg(frames=('image','nunique'), boxes=('image','size'), rotated=('rotated','sum')).to_string())
print(B.groupby(['clip','category']).size().unstack(fill_value=0).to_string())
# ---- end of Phase 1 code ----

In [ ]:
# Phase 1 inputs and output: same zips in the same order, and B == box columns of the Phase 1 per-box file
check('Phase 1 FILES == Al-Hada zip inputs, same order', [Path(IN[k]).name for k in AH_ZIPS], FILES)
check('Phase 1 AN == folder of the zip inputs', {rel(P[k].parent) for k in AH_ZIPS}, {rel(AN)})
P1 = pd.read_csv(P['ah_phase1_boxes'], usecols=PHASE1_BOX_COLS, keep_default_na=False)   # box columns only, no predictions
check('Phase 1 boxes file rows', EXPECT['ah_boxes'], len(P1))
check('B rows == Phase 1 boxes file rows', len(P1), len(B))
for col in ('file', 'image', 'category', 'rotated', 'clip'):
    check(f'{col} equal row by row (B vs Phase 1 file)', 0, int((B[col].astype(str).to_numpy() != P1[col].astype(str).to_numpy()).sum()))
b4, p4 = B[['x', 'y', 'w', 'h']].to_numpy(float), P1[['x', 'y', 'w', 'h']].to_numpy(float)
P1_MAXDIFF = float(np.abs(b4 - p4).max())
print(f'x, y, w, h: max abs diff vs Phase 1 file = {P1_MAXDIFF:.3e} px')
check('x, y, w, h == Phase 1 file (np.allclose, rtol=0, atol=1e-6)', 'allclose', f'max abs diff {P1_MAXDIFF:.3e}', ok=np.allclose(b4, p4, rtol=0, atol=1e-6))   # rtol=0: the default rtol=1e-5 would allow ~0.02 px at x=1900
del P1, b4, p4

## 2. Output folder (never overwritten) and the check log

In [ ]:
TODAY = time.strftime('%Y-%m-%d')
HAND_DIR = ROOT / '01_Data/Handoff'; HAND_DIR.mkdir(parents=True, exist_ok=True)
_done = [int(m.group(1)) for f in os.listdir(HAND_DIR) if (m := re.fullmatch(rf'{TODAY}_Track1Handoff_v(\d+)', f))]   # _INCOMPLETE folders do not count
VERSION = max(_done, default=0) + 1
def out_name(topic, ext=None): return f'{TODAY}_{topic}_v{VERSION}' + (f'.{ext}' if ext else '')
FINAL = HAND_DIR / out_name('Track1Handoff')
OUT = HAND_DIR / (FINAL.name + '_INCOMPLETE')
if OUT.exists():                                   # never deleted automatically
    print(f'a failed run left {OUT.name}: inspect and delete it manually, then re-run')
    raise FileExistsError(f'{rel(OUT)} exists')
OUT.mkdir(exist_ok=False)                          # renamed to FINAL only if every check passes
assert not FINAL.exists()
OUT_REL = FINAL.relative_to(ROOT).as_posix()
print('output folder:', OUT_REL, '(written as', OUT.name, 'until all checks pass)')

## 3. Split: byte-identical copy and `image_split.csv`

`splits_v1.json` also has `seadronessee.video_to_split`, `seadronessee.images`, `uavdt.seq_to_split`, `uavdt.joined_groups`, `uavdt.frame_step` and `uavdt.frames`. `group_to_split` is the key used here. Checked below: `uavdt.seq_to_split` and `seadronessee.video_to_split` each equal their `group_to_split` exactly, both `joined_groups` are empty, and `seadronessee.images` (split → list of image names) has no image under two splits and every image it lists is in `image_split` with the same split; SeaDronesSee images in `image_split` that it does not list are counted (info). Any other structure of `seadronessee.images` is printed and that check is marked skipped. SeaDronesSee images come from SDSClean (`split_group`), UAVDT frames from UAVDTFrames: **all** frames, every frame gets its sequence's split, `keep_5fps` copied as is (empty for SeaDronesSee). Track 1 crops come only from `keep_5fps` frames.

In [ ]:
shutil.copyfile(P['splits'], OUT / 'splits_v1.json')
check('splits_v1.json sha256 == source', sha256(P['splits']), sha256(OUT / 'splits_v1.json'))
S = json.load(open(P['splits']))
print('top-level keys:', list(S), '| seadronessee:', list(S['seadronessee']), '| uavdt:', list(S['uavdt']))
for d in ('seadronessee', 'uavdt'): assert 'group_to_split' in S[d], f'{d}.group_to_split missing'
check('seadronessee.joined_groups empty', 0, len(S['seadronessee']['joined_groups']))
check('uavdt.joined_groups empty', 0, len(S['uavdt']['joined_groups']))
check('uavdt.seq_to_split == uavdt.group_to_split (UAVDT groups are sequences)', S['uavdt']['group_to_split'], S['uavdt']['seq_to_split'])
check('seadronessee.video_to_split == seadronessee.group_to_split', S['seadronessee']['group_to_split'], S['seadronessee']['video_to_split'])
G2S = {'SeaDronesSee': S['seadronessee']['group_to_split'], 'UAVDT': S['uavdt']['group_to_split']}
for d, g in G2S.items():
    check(f'{d}: every group_to_split value is a split', True, set(g.values()) <= set(SPLITS))
check('no group name shared by SeaDronesSee and UAVDT', [], sorted(set(G2S['SeaDronesSee']) & set(G2S['UAVDT'])))

# SeaDronesSee: every image of SDSClean
J = json.load(open(P['sds_clean']))
sds = pd.DataFrame([{'image': im['file_name'], 'group': im['split_group']} for im in J['images']])
SDS_CATS = {c['id']: c['name'] for c in J['categories']}
del J
# UAVDT: ALL frames, same image format as ClassCrops ('<seq>/img<frame:06d>.jpg'); keep_5fps copied as is
uf = pd.read_csv(P['uav_frames'])
check('UAVDTFrames has seq, frame, keep_5fps', True, {'seq', 'frame', 'keep_5fps'} <= set(uf.columns))
check('UAVDTFrames rows (all frames)', EXPECT['uav_frames'], len(uf))
check('UAVDTFrames keep_5fps is boolean', 'bool', str(uf.keep_5fps.dtype))
check('UAVDTFrames keep_5fps frames', EXPECT['uav_frames_5fps'], int(uf.keep_5fps.sum()))
print(f"UAVDTFrames: {len(uf)} frames, {int(uf.keep_5fps.sum())} with keep_5fps (Track 1 crops come only from these)")
uav = pd.DataFrame({'image': uf.seq + '/img' + uf.frame.map('{:06d}'.format) + '.jpg', 'group': uf.seq, 'keep_5fps': uf.keep_5fps})

IS = pd.concat([sds.assign(dataset='SeaDronesSee'), uav.assign(dataset='UAVDT')], ignore_index=True)
IS['split'] = [G2S[d].get(g) for d, g in zip(IS.dataset, IS.group)]
IS['subset'] = np.where((IS.dataset == 'SeaDronesSee') & (IS.group == 'rgb'), 'rgb', IS.group)
check('image_split: every image has a split', 0, int(IS.split.isna().sum()))
check('image_split: image unique per dataset', 0, int(IS.duplicated(['dataset', 'image']).sum()))
check('image_split: no group in two splits', 0, int((IS.groupby(['dataset', 'group']).split.nunique() > 1).sum()))
rgb = IS[IS.subset == 'rgb']
check("image_split: SDS subset 'rgb' images", EXPECT['sds_rgb_images'], len(rgb))
check("image_split: all SDS 'rgb' images in test", ['test'], sorted(rgb.split.unique()))

# seadronessee.images: dict split -> list of image names
SI = S['seadronessee']['images']
print('seadronessee.images:', type(SI).__name__, '| keys:', list(SI) if isinstance(SI, dict) else '-')
if isinstance(SI, dict) and SI and all(isinstance(v, list) for v in SI.values()):
    check('seadronessee.images keys == splits', sorted(SPLITS), sorted(SI))
    check('seadronessee.images: every value is a list', True, all(isinstance(v, list) for v in SI.values()))
    inv = pd.DataFrame([(im, sp) for sp, ims in SI.items() for im in ims], columns=['image', 'split_images'])
    check('seadronessee.images: no image under two splits', 0, int((inv.groupby('image').split_images.nunique() > 1).sum()))
    inv = inv.drop_duplicates()                                         # image -> split
    sds_is = IS.loc[IS.dataset == 'SeaDronesSee', ['image', 'split']]
    j = inv.merge(sds_is, on='image', how='left')
    check('every seadronessee.images image is in image_split (SeaDronesSee)', 0, int(j.split.isna().sum()))
    check('seadronessee.images split == image_split split', 0, int((j.split != j.split_images).sum()))
    n_unlisted = int((~sds_is.image.isin(set(inv.image))).sum())
    REPORT.append({'check': 'SeaDronesSee image_split images not listed in seadronessee.images (info)', 'expected': None,
                   'actual': n_unlisted, 'pass': True})
    print(f'SeaDronesSee image_split images not listed in seadronessee.images (info): {n_unlisted}')
    per = {sp: int((inv.split_images == sp).sum()) for sp in SPLITS}
    SDS_IMAGES_CHECK = (f"ran: {len(inv)} images listed ({', '.join(f'{k} {v}' for k, v in per.items())}), no image under two splits, "
                        f"all in image_split with the same split; {n_unlisted} SeaDronesSee image_split images not listed there (info)")
    del inv, j, sds_is
else:
    first = next(iter(SI.values())) if isinstance(SI, dict) and SI else (SI[0] if isinstance(SI, list) and SI else None)
    print('seadronessee.images is not a dict split -> list of images; structure:', type(SI).__name__, len(SI) if hasattr(SI, '__len__') else '',
          '| first value type:', type(first).__name__, '| skipped')
    SDS_IMAGES_CHECK = 'skipped: not a dict split -> list of images'
    REPORT.append({'check': 'seadronessee.images split == image_split split', 'expected': 'dict split -> list of images',
                   'actual': type(SI).__name__, 'pass': True, 'status': 'skipped'})
del SI
uav_seqs = set(IS.loc[IS.dataset == 'UAVDT', 'group'])
check('image_split: UAVDT sequences', EXPECT['uav_sequences'], len(uav_seqs))
check('image_split: every UAVDT sequence in uavdt.group_to_split', [], sorted(uav_seqs - set(G2S['UAVDT'])))
check('image_split: UAVDT rows == UAVDTFrames rows', len(uf), int((IS.dataset == 'UAVDT').sum()))
IS = IS[['dataset', 'image', 'group', 'split', 'subset', 'keep_5fps']]
IS.to_csv(OUT / 'image_split.csv', index=False)
print(IS.groupby(['dataset', 'split']).size().unstack().to_string())

## 4. `crops_manifest.csv`: CropIndex_v1 rows unchanged + `npy_file`, `classifier_index`, `subset`

The `.npy` files are referenced, not copied; they are opened with `mmap_mode='r'` for the shape only.

In [ ]:
CI = pd.read_csv(P['crop_index'], keep_default_na=False, dtype={'ann_id': str})
CI_COLS = ['crop_id', 'split', 'row', 'dataset', 'category', 'image', 'x', 'y', 'w', 'h', 'box_side', 'size_bin',
           'group', 'source', 'track', 'ann_id', 'label_changed']
check('CropIndex columns', CI_COLS, list(CI.columns))
check('CropIndex rows', EXPECT['crops_total'], len(CI))
check('crop_id unique', True, bool(CI.crop_id.is_unique))
check('crops per split', EXPECT['crops_per_split'], {s: int((CI.split == s).sum()) for s in SPLITS})
check('crop categories', sorted(CLASSES), sorted(CI.category.unique()))

# per split x class == SplitClassTable_v1
T = pd.read_csv(P['split_table'], index_col=0)
ref = T.loc[CLASSES, SPLITS].astype(int)
got = CI.pivot_table(index='category', columns='split', values='crop_id', aggfunc='size', fill_value=0).reindex(index=CLASSES, columns=SPLITS).fillna(0).astype(int)
check('crops per split x class == SplitClassTable_v1', ref.to_dict(), got.to_dict())

# every crop group exists in group_to_split with the same split (ClassCrops and CropIndex)
CC = pd.read_csv(P['class_crops'], keep_default_na=False, dtype={'ann_id': str})
for name, D in (('ClassCrops', CC), ('CropIndex', CI)):
    g = pd.Series([G2S[d].get(x) for d, x in zip(D.dataset, D.group)], index=D.index)
    check(f'{name}: every group in group_to_split', 0, int(g.isna().sum()))
    check(f'{name}: split agrees with group_to_split', 0, int((g != D.split).sum()))
check('ClassCrops rows == CropIndex rows', len(CC), len(CI))

# npy shapes (memory-mapped, never loaded)
NPY = {s: IN[f'npy_{s}'] for s in SPLITS}
for s in SPLITS:
    a = np.load(P[f'npy_{s}'], mmap_mode='r'); n = a.shape[0]; shape = tuple(a.shape); del a
    check(f'{s}: max(row)+1 == npy rows', n, int(CI.loc[CI.split == s, 'row'].max()) + 1)
    check(f'{s}: rows 0..n-1 each used once', True, sorted(CI.loc[CI.split == s, 'row']) == list(range(n)))
    print(f'  {NPY[s]}: shape {shape}')

M = CI.copy()
M['npy_file'] = M.split.map(NPY)
M['classifier_index'] = M.category.map({c: i for i, c in enumerate(CLASSES)}).astype(int)
M['subset'] = np.where((M.dataset == 'SeaDronesSee') & (M.source == 'rgb'), 'rgb', M.group)   # same convention as image_split
check("crops_manifest: subset 'rgb' crops", EXPECT['rgb_crops'], int((M.subset == 'rgb').sum()))
check("crops_manifest: all subset 'rgb' crops in test", ['test'], sorted(M.loc[M.subset == 'rgb', 'split'].unique()))
rgb_img = (M.dataset == 'SeaDronesSee') & M.image.isin(set(IS.loc[IS.subset == 'rgb', 'image']))   # image_split: group == 'rgb'
check("crops subset 'rgb' (source == 'rgb') == crops of image_split 'rgb' images (group == 'rgb')", 0, int(((M.subset == 'rgb') != rgb_img).sum()))

ji = M[['dataset', 'image', 'split']].merge(IS[['dataset', 'image', 'split']], on=['dataset', 'image'], how='left', suffixes=('', '_img'))
check('every crop image is in image_split', 0, int(ji.split_img.isna().sum()))
check('crop split == image split', 0, int((ji.split != ji.split_img).sum()))
uk = M.loc[M.dataset == 'UAVDT', ['image']].merge(IS.loc[IS.dataset == 'UAVDT', ['image', 'keep_5fps']], on='image', how='left')
check('every UAVDT crop comes from a keep_5fps frame', 0, int((uk.keep_5fps != True).sum()))
M.to_csv(OUT / 'crops_manifest.csv', index=False)

## 5. `class_map.json`

Class order is the classifier's. The selected checkpoint is opened with `torch.load(..., weights_only=True)` only to read its class list; no model is built. If it stores a class list it must be identical; if it stores none, that is printed and the notebook continues.

In [ ]:
import torch
def skip_ckpt(why):
    global CKPT_CHECK
    CKPT_CHECK = f'skipped ({why})'
    REPORT.append({'check': 'checkpoint class list == CLASSES', 'expected': CLASSES, 'actual': why, 'pass': True, 'status': 'skipped'})
try:
    ck = torch.load(P['ckpt'], map_location='cpu', weights_only=True)    # tensors and plain types only; never weights_only=False
except Exception as e:
    ck = None; print(f'checkpoint class list not checked: {type(e).__name__}'); skip_ckpt(type(e).__name__)
if ck is not None:
    if isinstance(ck, dict) and 'classes' in ck:
        check('checkpoint class list == CLASSES', CLASSES, list(ck['classes']))   # a different list is a hard failure
        CKPT_CHECK = 'ran: stored class list == CLASSES'
    else:
        print('checkpoint stores no class list (skipped, not a failure)'); skip_ckpt('no class list stored')
del ck

sds_names = sorted(set(CI.loc[CI.dataset == 'SeaDronesSee', 'category']))
uav_names = sorted(set(CI.loc[CI.dataset == 'UAVDT', 'category']))
check('SDS crop classes are SDSClean category names', True, set(sds_names) <= set(SDS_CATS.values()))
AH_MAP = MAP                                                 # Phase 1 CVAT map (section 1b)
CLASS_MAP = {
    'classes': [{'classifier_index': i, 'coco_category_id': i + 1, 'name': c, 'domain': DOMAIN[c],
                 'source_labels': {'SeaDronesSee': [c] if c in sds_names else [],
                                   'UAVDT': [c] if c in uav_names else [],
                                   'AlHada_CVAT': [k for k, v in AH_MAP.items() if v == c]}}
                for i, c in enumerate(CLASSES)],
    'vehicle_rule': 'pickup -> truck; Hiace-type minibus -> bus; closed van -> car',
    'notes': ['coco_category_id = classifier_index + 1.',
              'UAVDT crops are capped per track (car 10, truck/bus 40 crops per track): the crops are NOT all UAVDT boxes.',
              'SeaDronesSee category ids in SDSClean are the dataset\'s own ids; use the names, not the ids.'],
}
check('class_map entries', 8, len(CLASS_MAP['classes']))
json.dump(CLASS_MAP, open(OUT / 'class_map.json', 'w'), indent=1)
CAT_ID = {c: i + 1 for i, c in enumerate(CLASSES)}

## 6. Al-Hada: frames, the three CVAT exports, merged COCO

Boxes come from `B` (section 1b, Phase 1 code). A second read-only pass over the same zips, in the same order, collects each export's image list and the original CVAT box and rotation (`rbox_cvat`); it must line up with `B` row by row.

In [ ]:
from PIL import Image
man = pd.read_csv(P['ah_manifest'], keep_default_na=False)
check('frames manifest rows', EXPECT['ah_manifest_rows'], len(man))
FRAME_RE = re.compile(r'AlHada_(CLIP0\d)_t(\d{4})\.jpg')
check("frames manifest: column 'frame' present", True, 'frame' in man.columns)
check("frames manifest: every 'frame' is AlHada_CLIP0X_tNNNN.jpg", 0, int((~man.frame.astype(str).str.fullmatch(FRAME_RE.pattern)).sum()))
MAN = pd.DataFrame({'file_name': man.frame.astype(str)})
check('frames manifest: file names unique', True, bool(MAN.file_name.is_unique))
FR = P['ah_frames']

# second pass: image list per export + original CVAT box and rotation, same zips and order as Phase 1 FILES
IMGS, RAW = {}, []
for zf in FILES:
    with zipfile.ZipFile(f'{AN}/{zf}') as zz:
        js = [n for n in zz.namelist() if n.endswith('.json')]
        check(f'{zf}: exactly one .json inside', 1, len(js))
        cj = json.loads(zz.read(js[0]))
    check(f'{zf}: CVAT labels are Phase 1 MAP keys', True, {k['name'] for k in cj['categories']} <= set(MAP))
    names = {i['id']: i['file_name'] for i in cj['images']}
    labels = {k['id']: MAP[k['name']] for k in cj['categories']}
    for i in cj['images']: IMGS.setdefault(i['file_name'], []).append(zf)
    for a in cj['annotations']:
        rot = (a.get('attributes') or {}).get('rotation', 0) or 0          # same rule as Phase 1
        RAW.append({'file': zf, 'image': names[a['image_id']], 'category': labels[a['category_id']],
                    'rbox_cvat': [float(v) for v in a['bbox']] + [float(rot)]})
RAW = pd.DataFrame(RAW)
check('second pass rows == B rows', len(B), len(RAW))
check('second pass (file, image, category) == B row by row', 0,
      int((RAW[['file', 'image', 'category']].to_numpy() != B[['file', 'image', 'category']].to_numpy()).any(axis=1).sum()))
check('export image names are AlHada_CLIP0X_tNNNN.jpg (no folder)', [], sorted(n for n in IMGS if not FRAME_RE.fullmatch(n)))
check('B clip == clip in the file name', 0, int((B.image.str.extract(FRAME_RE.pattern)[0] != B['clip']).sum()))

A = pd.DataFrame({'file_name': B.image, 'clip': B['clip'], 'category': B.category, 'source_zip': B.file,
                  'bbox_phase1': B[['x', 'y', 'w', 'h']].to_numpy(float).tolist(), 'rbox_cvat': RAW.rbox_cvat})
ann_frames = sorted(set(A.file_name))                      # annotated = at least one box
check('each frame in exactly one export', [], sorted(fn for fn, z in IMGS.items() if len(z) > 1))
no_box = sorted(set(IMGS) - set(ann_frames))
excl_names = {f'AlHada_{c}_t{t:04d}.jpg' for c, t in EXCLUDED}
check('frames in the exports without any box are among the excluded frames', [], sorted(set(no_box) - excl_names))
print(f'frames in the exports without boxes: {len(no_box)} (all listed as excluded)')
check('Al-Hada boxes', EXPECT['ah_boxes'], len(A))
check('annotated frames per clip', EXPECT['ah_frames_per_clip'],
      {k: int(v) for k, v in pd.Series(ann_frames).str[7:13].value_counts().sort_index().items()})
check('boxes per clip x class', EXPECT['ah_boxes_per_clip_class'],
      {cl: {k: int(v) for k, v in g.category.value_counts().reindex(['car', 'truck', 'bus']).fillna(0).items()} for cl, g in A.groupby('clip')})
check('boxes per class', EXPECT['ah_boxes_per_class'],
      {k: int(v) for k, v in A.category.value_counts().reindex(['car', 'truck', 'bus']).fillna(0).items()})
check('no duplicate (image, x, y, w, h)', 0, int(B.duplicated(['image', 'x', 'y', 'w', 'h']).sum()))

In [ ]:
# frames: exist, 1920x1080, inside Team_AlHada/Frames; manifest = annotated + excluded, disjoint
IMG_W, IMG_H = EXPECT['ah_image_size']               # not W, H: Phase 1 code uses those names
sizes = {}
for fn in ann_frames:
    p = FR / fn
    assert p.resolve().parent == FR.resolve(), f'path outside Team_AlHada/Frames: {fn}'
    assert p.exists(), f'frame missing: {rel(p)}'
    with Image.open(p) as im: sizes[fn] = im.size
check('annotated frames exist in Frames/', len(ann_frames), len(sizes))
check('all annotated frames are 1920x1080', [(IMG_W, IMG_H)], sorted(set(sizes.values())))
excl = {f'AlHada_{c}_t{t:04d}.jpg' for c, t in EXCLUDED}
check('excluded frames are in the manifest', True, excl <= set(MAN.file_name))
check('annotated and excluded frames are disjoint', [], sorted(excl & set(ann_frames)))
check('annotated + excluded == manifest (symmetric difference)', [], sorted(set(MAN.file_name) ^ (set(ann_frames) | excl)))

# clip to the image; report overshoot
def clip_box(b):
    x0, y0, x1, y1 = b[0], b[1], b[0] + b[2], b[1] + b[3]
    cx0, cy0, cx1, cy1 = max(0.0, x0), max(0.0, y0), min(float(IMG_W), x1), min(float(IMG_H), y1)
    over = max(0.0, -x0, -y0, x1 - IMG_W, y1 - IMG_H)
    if over == 0: return [float(v) for v in b], 0.0      # inside the image: bbox == bbox_phase1 exactly
    return [cx0, cy0, cx1 - cx0, cy1 - cy0], over
cl = A.bbox_phase1.map(clip_box)
A['bbox'] = cl.str[0]; A['overshoot_px'] = cl.str[1]
n_clipped = int((A.overshoot_px > 0).sum())
REPORT.append({'check': 'boxes clipped to the image (info)', 'expected': None,
               'actual': {'clipped': n_clipped, 'max_overshoot_px': float(A.overshoot_px.max())}, 'pass': True})
print(f'clipped boxes: {n_clipped}, max overshoot {A.overshoot_px.max():.2f} px')
check('all boxes w,h > 0 after clipping', 0, int(A.bbox.map(lambda b: b[2] <= 0 or b[3] <= 0).sum()))

In [ ]:
# merged COCO
img_ids = {fn: i + 1 for i, fn in enumerate(ann_frames)}
FR_REL = rel(FR)
COCO_OUT = {
    'info': {'description': 'Al-Hada team footage, CVAT boxes merged by Track 1 handoff', 'date_created': TODAY,
             'phase1_commit': PHASE1_COMMIT, 'note': 'blurred frames only, evaluation only',
             'file_name_relative_to': FR_REL,
             'source_zips': [Path(IN[k]).name for k in AH_ZIPS]},
    'images': [{'id': img_ids[fn], 'file_name': fn, 'width': sizes[fn][0], 'height': sizes[fn][1], 'clip': fn[7:13]}
               for fn in ann_frames],
    'annotations': [{'id': i + 1, 'image_id': img_ids[r.file_name], 'category_id': CAT_ID[r.category],
                     'bbox': [float(v) for v in r.bbox], 'bbox_phase1': [float(v) for v in r.bbox_phase1],
                     'rbox_cvat': [float(v) for v in r.rbox_cvat], 'area': float(r.bbox[2] * r.bbox[3]), 'iscrowd': 0,
                     'source_zip': r.source_zip} for i, r in enumerate(A.itertuples())],
    'categories': [{'id': CAT_ID[c], 'name': c, 'supercategory': DOMAIN[c]} for c in CLASSES],
}
json.dump(COCO_OUT, open(OUT / 'alhada_coco.json', 'w'))
check('COCO images', sum(EXPECT['ah_frames_per_clip'].values()), len(COCO_OUT['images']))
check('COCO annotations', EXPECT['ah_boxes'], len(COCO_OUT['annotations']))
check('COCO categories', 8, len(COCO_OUT['categories']))
coco = COCO(str(OUT / 'alhada_coco.json'))
check('pycocotools loads alhada_coco.json', len(COCO_OUT['annotations']), len(coco.getAnnIds()))

ex = pd.DataFrame([{'clip': c, 'file_name': f'AlHada_{c}_t{t:04d}.jpg', 'reason': EXCLUDED_REASON,
                    'note': 'not an empty negative; do not use as one'} for c, t in EXCLUDED])
ex.to_csv(OUT / 'excluded_frames.csv', index=False)

## 7. Preview: 10 random annotated frames (seed 42), boxes and class names drawn

In [ ]:
from PIL import ImageDraw
PRE = OUT / 'preview'; PRE.mkdir()
COLOR = {'car': (0, 200, 255), 'truck': (255, 160, 0), 'bus': (255, 0, 120)}
pick = np.random.default_rng(PREVIEW_SEED).choice(len(ann_frames), PREVIEW_N, replace=False)
for i in sorted(pick):
    fn = ann_frames[i]
    with Image.open(FR / fn) as im:
        im = im.convert('RGB'); d = ImageDraw.Draw(im)
        for r in A[A.file_name == fn].itertuples():
            x, y, w, h = r.bbox; d.rectangle([x, y, x + w, y + h], outline=COLOR[r.category], width=2)
            d.text((x, max(0, y - 11)), r.category, fill=COLOR[r.category])
        im.save(PRE / fn.replace('.jpg', '_preview.jpg'), quality=90)
print('preview:', sorted(p.name for p in PRE.iterdir()))

## 8. README

In [ ]:
n_split = IS.groupby(['dataset', 'split']).size().unstack().reindex(columns=SPLITS).fillna(0).astype(int)
README = f"""# Track 1 data handoff ({TODAY}, v{VERSION})

Labels only, no test evaluation, no TestLog entry, no predictions. No model was built (the selected checkpoint was opened with weights_only=True only to compare its class list).\nCheckpoint class-list check ({IN['ckpt']}): {CKPT_CHECK}.
All paths are relative to the project Drive folder. Phase 1 commit: {PHASE1_COMMIT}.

## Contents
| file | content | rows |
|---|---|---|
| splits_v1.json | byte-identical copy of {IN['splits']} | - |
| image_split.csv | dataset, image, group, split, subset, keep_5fps for every SeaDronesSee image and every UAVDT frame (keep_5fps as in UAVDTFrames; empty for SeaDronesSee) | {len(IS)} |
| crops_manifest.csv | {IN['crop_index']} unchanged + npy_file, classifier_index, subset | {len(M)} |
| class_map.json | 8 classes: classifier_index, coco_category_id, name, domain, source labels | 8 |
| alhada_coco.json | Al-Hada CVAT boxes, merged COCO (file_name relative to {FR_REL}) | {len(COCO_OUT['images'])} images, {len(COCO_OUT['annotations'])} boxes |
| excluded_frames.csv | extracted Al-Hada frames with no labels ({EXCLUDED_REASON}) | {len(ex)} |
| preview/ | {PREVIEW_N} random Al-Hada frames (seed {PREVIEW_SEED}) with boxes | {PREVIEW_N} |
| validation_report.json | every check: expected / actual / pass | - |
| checksums.sha256 | sha256 of every file above | - |

## Counts
Images per split:
| dataset | {' | '.join(SPLITS)} |
|---|{'---|' * len(SPLITS)}
{chr(10).join(f"| {d} | {' | '.join(str(v) for v in r)} |" for d, r in n_split.iterrows())}

Crops per split: {', '.join(f'{s} {int((M.split == s).sum())}' for s in SPLITS)} (total {len(M)}). The .npy crop arrays are referenced in `npy_file`, not copied; `row` is the index into that array.
Al-Hada: {len(COCO_OUT['images'])} annotated frames ({', '.join(f'{k} {v}' for k, v in EXPECT['ah_frames_per_clip'].items())}), {len(A)} boxes (car {EXPECT['ah_boxes_per_class']['car']}, truck {EXPECT['ah_boxes_per_class']['truck']}, bus {EXPECT['ah_boxes_per_class']['bus']}).

## Classes
Order: {', '.join(CLASSES)} (classifier_index 0-7, coco_category_id = index + 1).
Vehicle rule: pickup -> truck; Hiace-type minibus -> bus; closed van -> car.
UAVDT crops are capped per track (car 10, truck/bus 40): the crops are not all UAVDT boxes.
Track 1 crops come only from keep_5fps frames.

## Split key
Use `group_to_split` in splits_v1.json (`seadronessee` and `uavdt`). The file also has seadronessee.video_to_split, seadronessee.images, uavdt.seq_to_split, uavdt.joined_groups, uavdt.frame_step and uavdt.frames. Checked by this handoff: uavdt.seq_to_split == uavdt.group_to_split and seadronessee.video_to_split == seadronessee.group_to_split exactly; seadronessee.joined_groups and uavdt.joined_groups are empty; seadronessee.images: {SDS_IMAGES_CHECK}.
SeaDronesSee group 'rgb' (20 MP trinity stills) is all in test; `subset == 'rgb'` marks it so the "test without rgb" headline can be reproduced.

## Rules
- The split is never modified. Detection and segmentation use the same group_to_split, so no flight or sequence is in two splits across tracks.
- Test is for final evaluation only, never for model selection or tuning.
- Al-Hada is evaluation only: never trained or tuned on (dataset card).
- Al-Hada = blurred frames only; the raw videos stay owner-only and are never copied here or into the repo.
- excluded_frames.csv lists frames with the camera blocked: they are not empty negatives and must not be used as such.
- `bbox_phase1` was checked against the box columns of {PHASE1_BOXES} (max abs diff {P1_MAXDIFF:.2e} px; its prediction columns were not read).
- `bbox` in alhada_coco.json is the Phase 1 axis-aligned box clipped to the image; `bbox_phase1` is the unclipped Phase 1 box; `rbox_cvat` is the original CVAT [x, y, w, h, rotation_deg].
"""
open(OUT / 'README.md', 'w').write(README)

## 9. Hygiene, validation report, checksums, summary

In [ ]:
BAD = ['/content/', '/Users/', 'MyDrive']
TEXT_EXT = {'.json', '.csv', '.md', '.txt', '.sha256'}
hits = []
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p.suffix in TEXT_EXT:
        t = p.read_text(errors='ignore')
        hits += [f'{rel(p)}: {b}' for b in BAD if b in t]
check('no absolute paths in output files', [], hits)

IN_SHA = {k: {'path': v, 'sha256': sha256(P[k]) if P[k].is_file() else None} for k, v in IN.items()}
def report_json():
    return json.dumps({'created': time.strftime('%Y-%m-%d %H:%M'), 'output': OUT_REL, 'phase1_commit': PHASE1_COMMIT,
                       'inputs': IN_SHA, 'checks': REPORT, 'all_pass': all(r['pass'] for r in REPORT),
                       'skipped': [r['check'] for r in REPORT if r.get('status') == 'skipped']}, indent=1, default=str)
check('no absolute paths in validation_report.json', [], [b for b in BAD if b in report_json()])   # scanned before writing
VR = report_json(); assert not any(b in VR for b in BAD)
(OUT / 'validation_report.json').write_text(VR)
files = sorted(p for p in OUT.rglob('*') if p.is_file() and p.name != 'checksums.sha256')
with open(OUT / 'checksums.sha256', 'w') as f:
    for p in files: f.write(f'{sha256(p)}  {p.relative_to(OUT).as_posix()}\n')
cs_hits = [b for b in BAD if b in (OUT / 'checksums.sha256').read_text()]   # scanned after writing (validation_report is already sealed)
print(f"{'PASS' if not cs_hits else 'FAIL'}  no absolute paths in checksums.sha256: {cs_hits}")
assert not cs_hits, f'absolute path in checksums.sha256: {cs_hits}'

def nrows(p):
    if p.suffix == '.csv': return len(pd.read_csv(p))
    if p.name == 'alhada_coco.json': return f"{len(COCO_OUT['images'])} img / {len(COCO_OUT['annotations'])} ann"
    if p.name == 'class_map.json': return len(CLASS_MAP['classes'])
    if p.name == 'validation_report.json': return f'{len(REPORT)} checks'
    return ''
summ = pd.DataFrame([{'file': p.relative_to(OUT).as_posix(), 'rows/entries': nrows(p), 'sha256[:12]': sha256(p)[:12]}
                     for p in files + [OUT / 'checksums.sha256']])
print(summ.to_string(index=False))
print(); print(pd.DataFrame(REPORT).assign(result=lambda d: [('SKIPPED' if r.get('status') == 'skipped' else 'PASS' if r['pass'] else 'FAIL') for r in REPORT])[['check', 'result']].to_string(index=False))
assert all(r['pass'] for r in REPORT) and not FINAL.exists()
OUT.rename(FINAL)
n_skip = sum(r.get('status') == 'skipped' for r in REPORT)
print(f'\nALL {len(REPORT) - n_skip} CHECKS PASS, {n_skip} skipped -> {OUT_REL}')